In [48]:
import pandas as pd
import numpy as np
import math
import text_utils
import heapq
from sklearn.metrics import confusion_matrix, classification_report
import json

In [49]:
df = pd.read_csv("/Users/abhinavarora/Desktop/Android Spam Classifier/Machine Learning/Data Sets /final_balanced.csv")
print(df)


# v1 = label (spam/ham), v2 = text message
y = df['Class'].values  # labels
x = df['Text'].values  # raw text messages

rows = len(x)

# Shuffle and split
np.random.seed(42)  
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x[train_idx]  
x_test = x[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

      Class                                               Text
0       ham  I jus reached home. I go bathe first. But my s...
1      spam  Zelle Alert — Your linked bank account has bee...
2      spam  Welcome to Select, an O2 service with added be...
3       ham    Yo, any way we could pick something up tonight?
4      spam  healthy reproductive life some testimonials : ...
...     ...                                                ...
18387  spam  Your Facebook account has been reported. Verif...
18388  spam  HDFC Bank: Your loan EMI is due. Pay: hdfc-emi...
18389  spam      Free 10,000 Rupees cash! Claim: cash10k[.]top
18390  spam  Your Netflix account has been locked. Reactiva...
18391  spam          Win iPhone 12! Click: iphone12-win[.]info

[18392 rows x 2 columns]


In [50]:
#A map that stores words in the format word:index
vocab_list = {}
#len(vocab_list)
vocab_size = 20000


#A simple tokenization of the text

#Creating tokens for training and tetsing which will later be vectorized
train_tokens = [text_utils.tokenize(x_train[i]) for i in range(len(x_train))]
test_tokens = [text_utils.tokenize(x_test[j]) for j in range(len(x_test))]

vocab_list = text_utils.build_vocab_list(vocab_size, x_train, train_tokens)
idf = text_utils.build_idf(x_train, train_tokens, vocab_list)


#Creating vectorized inputs for each training example in x_train
vectorized_inputs = [[0] * vocab_size for _ in range(len(x_train))]

vectorized_inputs = text_utils.vectorize(x_train, vocab_size, train_tokens, vocab_list, True, idf)

#Calculating the class ratios
class_ratio_map = text_utils.calculate_class_ratio(y_train)

In [51]:
#This will store the phi parameter used in the Multinomial Distribution as (k, j) where k corresponds to the class and j corresponds to the feature
phi_dict = {}
#phi j,k = (occurences of a feature exisiting given a specific class)/(number of examples of that specific class)

#Calculates the word count from the vectorized inputs for each ham and spam classes
def calculate_word_count(vectorized_x, training_y):
    res_dict = {}
    for i in range(len(vectorized_x)):
        for j in range(len(vectorized_x[0])):
            if training_y[i] not in res_dict:
                res_dict[training_y[i]] = vectorized_x[i][j]
            else:
                res_dict[training_y[i]] += vectorized_x[i][j]
    
    return res_dict

def calculate_phi_dict(vectorized_x, training_y):
    feature_number = 0
    total_words = calculate_word_count(vectorized_x, training_y)
    for i in range(len(vectorized_x[0])):
        auxilary_dict = {element:0 for element in class_ratio_map}
        for j in range(len(vectorized_x)):
            auxilary_dict[training_y[j]] += vectorized_x[j][i]
    
        for elt in auxilary_dict:
            #Laplace smoothing to ensure log(0) probabilities never appear
            phi_dict[str(elt) + str(feature_number)] = (auxilary_dict[elt] + 1)/(total_words[elt] + vocab_size)
        
        feature_number += 1

calculate_phi_dict(vectorized_inputs, y_train)

n_train = len(y_train)

def predict_one(test_vector):
    heap = []
    heapq.heapify_max(heap)
    for element in class_ratio_map:
        s = 0
        for i in range(len(test_vector)):
            s += test_vector[i] * math.log(phi_dict[str(element) + str(i)])
        heapq.heappush_max(heap, (math.log(class_ratio_map[element]/n_train) + s, element))
    
    return heap[0][1]


#Vectorizing test inputs
vectorized_test_inputs = [[0] * 1000 for _ in range(len(x_test))]
vectorized_test_inputs = text_utils.vectorize(x_test, vocab_size, test_tokens, vocab_list, True, idf)

y_pred = []

#Making predictions
def make_all_predictions(test_x, test_y):
    accuracy = 0
    ham_count = 0
    spam_count = 0
    for i in range(len(test_x)):
        prediction = predict_one(test_x[i])
        y_pred.append(prediction)
        if(prediction == test_y[i]):
            accuracy += 1

        if prediction == 'ham':
            ham_count += 1
        else:
            spam_count += 1

    print(accuracy/len(test_x))

make_all_predictions(vectorized_test_inputs, y_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Model Performance (vocab_size=10000): Accuracy=94.8%, Spam Precision=0.93, Spam Recall=0.97, Ham Precision=0.97, Ham Recall=0.92, F1=0.95
    

0.9431910845338407
[[1628  100]
 [ 109 1842]]
              precision    recall  f1-score   support

         ham       0.94      0.94      0.94      1728
        spam       0.95      0.94      0.95      1951

    accuracy                           0.94      3679
   macro avg       0.94      0.94      0.94      3679
weighted avg       0.94      0.94      0.94      3679



In [52]:
#Code to add parameters:
if True:
    parameter_dict = {
        "IDF": idf,
        "Phi dict": phi_dict,
        "Class ratios": class_ratio_map,
        "Train examples length": n_train,
        "Vocab size": vocab_size,
        "Vocab list": vocab_list
    }
    with open("model_params.json", "a") as params:
        json.dump(parameter_dict, params, indent=4)
